In [38]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_START_YEAR, DECOUPLING_END_YEAR

In [ ]:
# Emissions + urbanization for analysis
query = """
    SELECT
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_per_capita,
        e.co2_total,
        e.co2_per_gdp,
        e.consumption_co2_per_capita,
        e.gdp,
        e.population,
        u.urban_population_pct,
        u.urban_growth_rate,
        u.population_density,
        u.urban_population_total,
        u.pm25_exposure,
        u.electricity_access_pct,
        u.slum_population_pct
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    LEFT JOIN urbanization u 
        ON u.country_id = e.country_id 
        AND u.year = e.year
    WHERE e.year BETWEEN :start AND :end
    ORDER BY c.iso_code, e.year
"""

with engine.connect() as conn:
    df = pd.read_sql(
        text(query),
        conn,
        params={'start': DECOUPLING_START_YEAR, 'end': DECOUPLING_END_YEAR}
    )

df_eu = df[
    df['iso_code'].isin(EUROPEAN_COUNTRIES) &
    (df['iso_code'] != 'MLT')
].copy()

# GDP per capita
df['gdp_per_capita'] = df['gdp'] / df['population']
df_eu['gdp_per_capita'] = df_eu['gdp'] / df_eu['population']

print(f"Global: {len(df)} rows, {df['iso_code'].nunique()} countries")
print(f"Europe: {len(df_eu)} rows, {df_eu['iso_code'].nunique()} countries")
print(f"\nUrbanization data coverage:")
print(f" urban_population_pct: {df['urban_population_pct'].notna().sum()} rows")
print(f" population_density: {df['population_density'].notna().sum()} rows")
print(f" pm25_exposure: {df['pm25_exposure'].notna().sum()} rows")

2026-06-08 15:57:49,005 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-08 15:57:49,007 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-08 15:57:49,011 INFO sqlalchemy.engine.Engine [cached since 4250s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x00000227EEA80E10>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-08 15:57:49,013 INFO sqlalchemy.engine.Engine 
    SELECT
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_per_capita,
        e.co2_total,

In [40]:
# Use latest year with good coverage for cross-country comparison
latest = df[
    (df['year'] == 2022) &
    (df['urban_population_pct'].notna()) &
    (df['co2_per_capita'].notna())
].copy()

latest_eu = df_eu[
    (df_eu['year'] == 2022) &
    (df_eu['urban_population_pct'].notna()) &
    (df_eu['co2_per_capita'].notna())
].copy()

print(f"Global snapshot 2022: {len(latest)} countries")
print(f"European snapshot 2022: {len(latest_eu)} countries")
print(f"\nUrbanization range: {latest['urban_population_pct'].min():.1f}% - {latest['urban_population_pct'].max():.1f}%")
print(f"CO2 per capita range: {latest['co2_per_capita'].min():.2f} - {latest['co2_per_capita'].max():.2f} tonnes")

Global snapshot 2022: 204 countries
European snapshot 2022: 33 countries

Urbanization range: 14.7% - 100.0%
CO2 per capita range: 0.06 - 37.89 tonnes


In [41]:
# Global view - color by GDP per capita (wealth as third variable)
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='co2_per_capita',
    color='gdp_per_capita',
    size='population',
    size_max=75,
    hover_name='country',
    color_continuous_scale='RdYlGn',
    title='Urbanization vs CO2 per Capita - Global (2022)<br>'
          '<sup>Color = GDP per capita | Size = population</sup>',
    labels={
        'urban_population_pct': 'Urban population (%)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'gdp_per_capita': 'GDP per capita'
    },
    height=1000
)

# Add trend line
fig.update_traces(marker=dict(opacity=0.9))
fig.show()

# Correlation
corr = latest[['urban_population_pct', 'co2_per_capita', 'gdp_per_capita']].corr()
print("\nCorrelation matrix:")
print(corr.round(3))


Correlation matrix:
                      urban_population_pct  co2_per_capita  gdp_per_capita
urban_population_pct                 1.000           0.476           0.617
co2_per_capita                       0.476           1.000           0.742
gdp_per_capita                       0.617           0.742           1.000


In [42]:
# Global view - but with consumption co2
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='consumption_co2_per_capita',
    color='gdp_per_capita',
    size='population',
    size_max=75,
    hover_name='country',
    color_continuous_scale='RdYlGn',
    title='Urbanization vs Consumption CO2 per Capita - Global (2022)<br>'
          '<sup>Color = GDP per capita | Size = population</sup>',
    labels={
        'urban_population_pct': 'Urban population (%)',
        'consumption_co2_per_capita': 'Consumption CO2 per capita (tonnes)',
        'gdp_per_capita': 'GDP per capita'
    },
    height=1000
)

# Add trend line
fig.update_traces(marker=dict(opacity=0.9))
fig.show()

# Correlation
corr = latest[['urban_population_pct', 'consumption_co2_per_capita', 'gdp_per_capita']].corr()
print("\nCorrelation matrix:")
print(corr.round(3))


Correlation matrix:
                            urban_population_pct  consumption_co2_per_capita  \
urban_population_pct                       1.000                       0.634   
consumption_co2_per_capita                 0.634                       1.000   
gdp_per_capita                             0.617                       0.801   

                            gdp_per_capita  
urban_population_pct                 0.617  
consumption_co2_per_capita           0.801  
gdp_per_capita                       1.000  


In [ ]:
# Key question: is the urbanization - CO2 link real,
# or is it just because richer countries are more urbanized?

# Split into income groups by GDP per capita quartiles
latest['income_group'] = pd.qcut(
    latest['gdp_per_capita'],
    q=5,
    labels=['Low income', 'Lower-middle', 'Middle', 'Upper-middle', 'High income']
)

fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='co2_per_capita',
    color='income_group',
    hover_name='country',
    facet_col='income_group',
    trendline='ols',
    title='Urbanization vs CO2 per Capita - by Income Group<br>'
          '<sup>Does the relationship hold within income groups?</sup>',
    labels={
        'urban_population_pct': 'Urban pop. (%)',
        'co2_per_capita': 'CO2 per capita (tonnes)'
    },
    height=500
)
fig.show()

latest['income_group'] = pd.qcut(
    latest['gdp_per_capita'],
    q=5,
    labels=['Low income', 'Lower-middle', 'Middle', 'Upper-middle', 'High income']
)
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='consumption_co2_per_capita',
    color='income_group',
    hover_name='country',
    facet_col='income_group',
    trendline='ols',
    title='Urbanization vs Consumption CO2 per Capita - by Income Group<br>'
          '<sup>Does the relationship hold within income groups?</sup>',
    labels={
        'urban_population_pct': 'Urban pop. (%)',
        'consumption_co2_per_capita': 'CO2 per capita (tonnes)'
    },
    height=500
)
fig.show()

# Correlation within each income group
print("Correlation (urban_pct vs co2_per_capita) by income group:")
for group in latest['income_group'].cat.categories:
    subset = latest[latest['income_group'] == group]
    corr = subset['urban_population_pct'].corr(subset['co2_per_capita'])
    print(f"  {group}: {corr:.3f} (n={len(subset)})")

# Correlation within each income group
print("\nCorrelation (urban_pct vs consumption_co2_per_capita) by income group:")
for group in latest['income_group'].cat.categories:
    subset = latest[latest['income_group'] == group]
    corr = subset['urban_population_pct'].corr(subset['consumption_co2_per_capita'])
    print(f"  {group}: {corr:.3f} (n={len(subset)})")

Correlation (urban_pct vs co2_per_capita) by income group:
  Low income: 0.557 (n=33)
  Lower-middle: 0.221 (n=32)
  Middle: -0.072 (n=33)
  Upper-middle: -0.141 (n=32)
  High income: 0.342 (n=33)

Correlation (urban_pct vs consumption_co2_per_capita) by income group:
  Low income: 0.698 (n=33)
  Lower-middle: 0.328 (n=32)
  Middle: 0.086 (n=33)
  Upper-middle: -0.189 (n=32)
  High income: 0.515 (n=33)


In [85]:
# Population density vs CO2 per capita
# Theory: denser cities = more efficient = less emissions per person

density_df = latest[latest['population_density'].notna()].copy()

fig = px.scatter(
    density_df,
    x='population_density',
    y='co2_per_capita',
    color='urban_population_pct',
    hover_name='country',
    size='population',
    size_max=75,
    color_continuous_scale='Reds',
    title='Population Density vs CO2 per Capita (2022)<br>'
          '<sup>Do denser countries emit less per person?</sup>',
    labels={
        'population_density': 'Population density (people/km²)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'urban_population_pct': 'Urban<br>population (%)',
    },
    height=700,
    log_x=True  # log scale - density varies hugely
)
fig.show()

# European view - less noise, more comparable countries
fig_eu = px.scatter(
    latest_eu[latest_eu['population_density'].notna()],
    x='population_density',
    y='co2_per_capita',
    color='urban_population_pct',
    text='country',
    color_continuous_scale='reds',
    title='Density vs CO2 - Europe only (2022)',
    labels={
        'population_density': 'Population density (people/km²)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'urban_population_pct': 'Urban<br>population (%)'
    },
    height=700
)

custom_positions = {
    "Croatia": "bottom center",
    "Sweden": "bottom center",
    "Romania": "middle left",
    "Lithuania": "middle left",
    "Greece": "middle left",
    "Spain": "middle left",
    "Netherlands": "top left",
}
positions = [
    custom_positions.get(country, "middle right")
    for country in latest_eu["country"]
]
fig_eu.update_traces(textposition=positions)
fig_eu.show()

corr_density = density_df['population_density'].corr(density_df['co2_per_capita'])
print(f"\nGlobal correlation (density vs co2_per_capita): {corr_density:.3f}")
corr_eu_density = latest_eu['population_density'].corr(latest_eu['co2_per_capita'])
print(f"\nEurope correlation (density vs co2_per_capita): {corr_eu_density:.3f}")


Global correlation (density vs co2_per_capita): 0.005

Europe correlation (density vs co2_per_capita): 0.283
